# 03 - Detection evaluation and calibration

Run `make eval-detection` first. Every number comes from `evaluation/detection.py`
(`data/reports/detection_eval.json`, also logged to MLflow). This notebook only plots them.

In [ ]:
import json

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from core.settings import get_settings

settings = get_settings()
result = json.loads((settings.reports_dir / "detection_eval.json").read_text(encoding="utf-8"))
summary, scenarios = result["summary"], pd.DataFrame(result["scenarios"])
clean = pd.DataFrame(result["clean_anomalies"])
{k: summary[k] for k in ("recall", "recall_magnitude_ge_20pct", "false_alarms_per_30_days", "precision_fault_caused", "median_delay_days")}

## Recall by measured magnitude and by fault type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
pd.DataFrame(summary["recall_by_band"]).T["mean"].plot.bar(ax=axes[0], title="Recall by measured magnitude")
pd.DataFrame(summary["recall_by_type"]).T["mean"].plot.bar(ax=axes[1], title="Recall by fault type")
for ax in axes: ax.axhline(0.8, ls="--", c="grey"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()
scenarios[["id", "dimension", "value", "measured_magnitude", "detected", "delay_days"]]

## Measured magnitude vs detection

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for detected, g in scenarios.groupby("detected"):
    ax.scatter(g["measured_magnitude"], g["fault_type"], label="detected" if detected else "missed", marker="o" if detected else "x")
ax.axvline(0.2, ls="--", c="grey"); ax.set_xscale("log"); ax.set_xlabel("|relative change| in the targeted segment")
ax.legend(); plt.show()

## Clean-run alarms on the timeline (2018, no injected fault)

In [ ]:
con = duckdb.connect(str(settings.duckdb_path), read_only=True)
orders = con.execute("SELECT date, value FROM metrics.series WHERE kpi='orders' AND dimension='total' "
                     "AND date >= DATE '2018-01-01' ORDER BY date").df()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(orders["date"], orders["value"], lw=0.8, c="k")
if not clean.empty:
    for d in pd.to_datetime(clean["date"]).unique():
        ax.axvline(d, c="red", alpha=0.3)
ax.set_title(f"Total orders per day; red = clean-run alarm on any KPI ({len(clean)})"); plt.show()
clean